# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed for the current kernel
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset's Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a Dataset object, not a dict/list.

print(f"Dataset name: {metadata.name}")
print(f"Citation: {metadata.citeAs}")
print(f"Description: {metadata.description}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets and fields. All entities are referenced by their `@id` fields.

In [ ]:
# List available record sets and fields, referencing all by @id.
record_set_ids = [r['@id'] for r in metadata.to_json().get('recordSet', [])]
if not record_set_ids:
    print("No record sets declared in the metadata. Attempting to infer from distribution...")
    # Many Croissant datasets put the main data in a distribution, 
    # and mlcroissant often exposes a default record set as well.
    from pprint import pprint
    try:
        for rec_set in dataset.record_sets:
            print(f"RecordSet: {rec_set.id} (name: {rec_set.name})")
            if hasattr(rec_set, 'fields'):
                for field in rec_set.fields:
                    print(f"  Field: {field.id} (name: {field.name}, dataType: {field.dataType})")
    except Exception as e:
        print(f"Exception while listing record sets: {e}")
    # For reference in later steps, try to get the list of record set IDs
    record_set_ids = [rec_set.id for rec_set in dataset.record_sets]
else:
    print("Found explicit record sets: ", record_set_ids)
    for record_set_id in record_set_ids:
        print(f"RecordSet: {record_set_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All accesses use `@id` for record set and field references as required.

In [ ]:
# Extract data from each record set
# Use the inferred or known record set ids from above.

dataframes = {}

if not record_set_ids:
    print("No record sets detected. Cannot extract records.")
else:
    for record_set_id in record_set_ids:
        print(f"Loading records for record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        # Show columns for the first record set only
        if record_set_id == record_set_ids[0]:
            print(f"Columns for record set {record_set_id}:")
            print(df.columns.tolist())
            print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps using only `@id`s. Example: Remove outliers, normalize numeric fields, or group data by key fields.

In [ ]:
# Choose a record set and numeric field for demonstration.
if not record_set_ids:
    print("No record sets/data for EDA.")
else:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]

    # Attempt to infer a numeric field by @id name for example
    # List columns for user reference
    print('Available columns:', df.columns.tolist())

    # Common mass spectrometry column hints
    candidate_fields = [c for c in df.columns if any(x in c.lower() for x in ['abund', 'mw', 'weight', 'coverage', 'count', 'number', 'score'])]
    if candidate_fields:
        numeric_field_id = candidate_fields[0]
        print(f"Selected example numeric field by @id: {numeric_field_id}")
    else:
        # Fallback to any numeric-looking column
        numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
        if numeric_cols:
            numeric_field_id = numeric_cols[0]
            print(f"Selected first numeric column: {numeric_field_id}")
        else:
            raise Exception("No suitable numeric field in the data.")

    # Filter on the numeric field
    threshold = df[numeric_field_id].median() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head(3))

    # Normalize the field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head(3))

    # Group by a non-numeric field if available
    non_numeric_fields = [c for c in df.columns if df[c].dtype == object and c != numeric_field_id]
    if non_numeric_fields:
        group_field_id = non_numeric_fields[0]
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id} (showing first 3 groups):")
        print(grouped_df.head(3))
    else:
        print("No appropriate field for grouping found.")

## 5. Visualization
Visualize data distributions or relationships between fields using only `@id` field references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_set_ids or not dataframes:
    print("No data loaded for visualization")
else:
    df = dataframes[record_set_ids[0]]
    # Use same field IDs as before
    if 'numeric_field_id' in locals():
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
        plt.title(f'Distribution of {numeric_field_id}')
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.show()
        # Try scatter with grouping field if available
        if 'group_field_id' in locals() and group_field_id in df.columns:
            # Only show top 10 category for readability
            top_cats = df[group_field_id].value_counts().index[:10]
            subset = df[df[group_field_id].isin(top_cats)]
            plt.figure(figsize=(8, 5))
            sns.boxplot(data=subset, x=group_field_id, y=numeric_field_id)
            plt.title(f'{numeric_field_id} by {group_field_id}')
            plt.xticks(rotation=45)
            plt.show()

## 6. Conclusion
This notebook demonstrated loading and analyzing a complex FAIR dataset using the `mlcroissant` library, referencing all entities by their `@id` fields for reproducibility and clarity. You can use the above workflow as a template for other Croissant datasets!